# DefectLens AI — Exploratory Data Analysis

This notebook explores the **MVTec Anomaly Detection (MVTec AD)** dataset:
- Dataset structure and class distribution
- Image statistics (size, channel distribution)
- Normal vs. defective sample comparison
- Defect type breakdown per category

**Prerequisites:** Download the MVTec AD dataset and place it under `data/mvtec/`.
See `data/README.md` for instructions.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import yaml

from defectlens.data.dataset import MVTecDataset
from defectlens.data.validation import validate_dataset

DATA_ROOT = Path('../data/mvtec')

MVTEC_CATEGORIES = [
    'bottle', 'cable', 'capsule', 'carpet', 'grid',
    'hazelnut', 'leather', 'metal_nut', 'pill', 'screw',
    'tile', 'toothbrush', 'transistor', 'wood', 'zipper'
]

print('DefectLens AI — Dataset EDA')
print(f'Data root: {DATA_ROOT.resolve()}')
print(f'Exists: {DATA_ROOT.exists()}')

In [ ]:
# --- Category Availability Check ---
if DATA_ROOT.exists():
    available = [c for c in MVTEC_CATEGORIES if (DATA_ROOT / c).exists()]
    missing = [c for c in MVTEC_CATEGORIES if c not in available]
    print(f'Available categories ({len(available)}): {available}')
    print(f'Missing categories  ({len(missing)}): {missing}')
else:
    print('[WARN] Dataset not downloaded yet. Follow data/README.md to download MVTec AD.')
    available = []

In [ ]:
# --- Dataset Statistics per Category ---
if available:
    stats = []
    for cat in available:
        train_ds = MVTecDataset(root=str(DATA_ROOT), category=cat, split='train')
        test_ds  = MVTecDataset(root=str(DATA_ROOT), category=cat, split='test')
        n_normal = sum(1 for i in range(len(test_ds)) if test_ds[i]['label'] == 0)
        n_defect = sum(1 for i in range(len(test_ds)) if test_ds[i]['label'] == 1)
        stats.append({
            'category': cat,
            'train_samples': len(train_ds),
            'test_total': len(test_ds),
            'test_normal': n_normal,
            'test_defective': n_defect,
        })
    
    print(f'{'Category':<15} {'Train':>8} {'Test':>8} {'Normal':>8} {'Defect':>8}')
    print('-' * 55)
    for s in stats:
        print(f"{s['category']:<15} {s['train_samples']:>8} {s['test_total']:>8} {s['test_normal']:>8} {s['test_defective']:>8}")
    
    total_train = sum(s['train_samples'] for s in stats)
    total_test = sum(s['test_total'] for s in stats)
    print('-' * 55)
    print(f"{'TOTAL':<15} {total_train:>8} {total_test:>8}")

In [ ]:
# --- Sample Visualization: Normal vs. Defective ---
if available:
    cat = available[0]
    test_ds = MVTecDataset(root=str(DATA_ROOT), category=cat, split='test')
    
    normals  = [i for i in range(len(test_ds)) if test_ds[i]['label'] == 0][:3]
    defects  = [i for i in range(len(test_ds)) if test_ds[i]['label'] == 1][:3]
    
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    fig.suptitle(f'DefectLens AI — {cat.upper()} Samples', fontsize=15, fontweight='bold')
    
    for j, idx in enumerate(normals):
        item = test_ds[idx]
        img = item['image'].permute(1, 2, 0).numpy()
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        axes[0, j].imshow(img)
        axes[0, j].set_title(f'Normal #{j+1}', color='green')
        axes[0, j].axis('off')
    
    for j, idx in enumerate(defects):
        item = test_ds[idx]
        img = item['image'].permute(1, 2, 0).numpy()
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        axes[1, j].imshow(img)
        axes[1, j].set_title(f"Defective #{j+1} ({item.get('defect_type','?')})", color='red')
        axes[1, j].axis('off')
    
    plt.tight_layout()
    plt.savefig('../outputs/eda_samples.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/eda_samples.png')

In [ ]:
# --- Class Balance Bar Chart ---
if available and 'stats' in dir():
    cats = [s['category'] for s in stats]
    normals_count = [s['test_normal'] for s in stats]
    defects_count = [s['test_defective'] for s in stats]
    
    x = np.arange(len(cats))
    width = 0.4
    fig, ax = plt.subplots(figsize=(16, 6))
    ax.bar(x - width/2, normals_count, width, label='Normal', color='#28a745', alpha=0.85)
    ax.bar(x + width/2, defects_count, width, label='Defective', color='#dc3545', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(cats, rotation=45, ha='right')
    ax.set_ylabel('Sample Count')
    ax.set_title('MVTec AD — Normal vs. Defective Distribution per Category', fontsize=13)
    ax.legend()
    ax.grid(axis='y', alpha=0.4)
    plt.tight_layout()
    plt.savefig('../outputs/eda_class_distribution.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/eda_class_distribution.png')